# Bài 3 — Tic-Tac-Toe: Minimax, Alpha-Beta và move ordering

Protocol tái lập: seed 42; mỗi lần đo reset node counter; thời gian chỉ tính search bằng `time.perf_counter()` (không gồm render hoặc tính bảng kiểm tra).

In [1]:
import math
import time
from functools import lru_cache

import numpy as np
import pandas as pd

from tictactoe import actions, empty_board, other, result, utility

SEED = 42
np.random.seed(SEED)

def make_board(marks):
    board = empty_board()
    for index, symbol in marks.items():
        board[index] = symbol
    return board

STATES = {
    "T1 — X sắp thắng": make_board({0: "x", 1: "o", 3: "o", 4: "x"}),
    "T2 — X cần hòa": make_board({0: "x", 1: "o", 2: "x", 4: "o"}),
    "T3 — bất lợi": make_board({0: "o", 2: "x", 8: "o"}),
    "T4 — bàn rỗng": empty_board(),
}

def board_text(board):
    return "/".join("".join(board[i:i+3]).replace(" ", "·") for i in range(0, 9, 3))


In [2]:
def ordered_actions(board, move_ordering=False):
    legal = actions(board)
    if not move_ordering:
        return legal
    # Centre, corners, then edges; deterministic tie-breaker makes timing comparable.
    priority = [4, 0, 2, 6, 8, 1, 3, 5, 7]
    return [move for move in priority if move in legal]

def exact_search(board, player="x", alpha_beta=True, move_ordering=False):
    """Minimax/alpha-beta with a fresh node counter and perf_counter timing."""
    nodes = 0
    started = time.perf_counter()

    def visit(state, to_move, alpha=-math.inf, beta=math.inf):
        nonlocal nodes
        nodes += 1
        terminal_value = utility(state, player)
        if terminal_value is not None:
            return terminal_value
        maximizing = to_move == player
        value = -math.inf if maximizing else math.inf
        for move in ordered_actions(state, move_ordering):
            child_value = visit(result(state, to_move, move), other(to_move), alpha, beta)
            if maximizing:
                value = max(value, child_value)
                alpha = max(alpha, value)
            else:
                value = min(value, child_value)
                beta = min(beta, value)
            if alpha_beta and alpha >= beta:
                break
        return value

    root_value = visit(board, player)
    elapsed_s = time.perf_counter() - started
    root_scores = []
    # Reuse exact child values only for reporting the complete set of optimal moves.
    # This is excluded from elapsed_s/node counts of the algorithm above.
    for move in ordered_actions(board, move_ordering):
        root_scores.append((move, exact_value(result(board, player, move), other(player), player)))
    best_moves = tuple(move for move, value in root_scores if value == root_value)
    return {"move": best_moves[0] if best_moves else None, "optimal_moves": best_moves,
            "value": root_value, "nodes": nodes, "elapsed_s": elapsed_s}

@lru_cache(maxsize=None)
def _cached_value(state_tuple, to_move, player):
    state = list(state_tuple)
    terminal_value = utility(state, player)
    if terminal_value is not None:
        return terminal_value
    values = [_cached_value(tuple(result(state, to_move, move)), other(to_move), player)
              for move in actions(state)]
    return max(values) if to_move == player else min(values)

def exact_value(state, to_move, player):
    return _cached_value(tuple(state), to_move, player)


## Bốn trạng thái chung

`·` là ô trống. Mọi hàng trong các bảng dưới đều đánh giá nước đi cho X, đúng cùng bộ T1–T4 của nhóm.

In [3]:
pd.DataFrame([{"Trạng thái": name, "Bàn cờ": board_text(board), "Nước đi": "X"}
              for name, board in STATES.items()])

,Trạng thái,Bàn cờ,Nước đi
0,T1 — X sắp thắng,xo·/ox·/···,X
1,T2 — X cần hòa,xox/·o·/···,X
2,T3 — bất lợi,o·x/···/··o,X
3,T4 — bàn rỗng,···/···/···,X


## Minimax và Alpha-Beta

`optimal_moves` được đối chiếu theo tập: trong trường hợp có nhiều nước tối ưu, không coi việc hai thuật toán chọn hai chỉ số khác nhau là lỗi.

In [4]:
records = []
for name, board in STATES.items():
    minimax = exact_search(board, alpha_beta=False, move_ordering=False)
    ab = exact_search(board, alpha_beta=True, move_ordering=False)
    assert minimax["value"] == ab["value"], (name, minimax, ab)
    assert set(minimax["optimal_moves"]) == set(ab["optimal_moves"]), (name, minimax, ab)
    for label, found in (("Minimax", minimax), ("Alpha-Beta", ab)):
        records.append({"Trạng thái": name, "Thuật toán": label,
                        "Nước đi": found["move"], "Tập nước tối ưu": found["optimal_moves"],
                        "Giá trị": found["value"], "Số nút": found["nodes"],
                        "Thời gian (ms)": round(found["elapsed_s"] * 1_000, 3)})
comparison = pd.DataFrame(records)
comparison

,Trạng thái,Thuật toán,Nước đi,Tập nước tối ưu,Giá trị,Số nút,Thời gian (ms)
0,T1 — X sắp thắng,Minimax,2,"(2, 6, 8)",1,190,9.494
1,T1 — X sắp thắng,Alpha-Beta,2,"(2, 6, 8)",1,61,3.156
2,T2 — X cần hòa,Minimax,7,"(7,)",0,206,10.616
3,T2 — X cần hòa,Alpha-Beta,7,"(7,)",0,101,4.982
4,T3 — bất lợi,Minimax,1,"(1, 3, 4, 5, 6, 7)",-1,941,46.387
5,T3 — bất lợi,Alpha-Beta,1,"(1, 3, 4, 5, 6, 7)",-1,238,11.843
6,T4 — bàn rỗng,Minimax,0,"(0, 1, 2, 3, 4, 5, 6, 7, 8)",0,549946,26517.475
7,T4 — bàn rỗng,Alpha-Beta,0,"(0, 1, 2, 3, 4, 5, 6, 7, 8)",0,18297,1008.932


In [5]:
assertion_table = comparison.pivot(index="Trạng thái", columns="Thuật toán", values=["Giá trị", "Tập nước tối ưu"])
print("✓ Với cả bốn trạng thái, Minimax và Alpha-Beta có cùng giá trị và cùng tập nước tối ưu.")
assertion_table

✓ Với cả bốn trạng thái, Minimax và Alpha-Beta có cùng giá trị và cùng tập nước tối ưu.


Giá trị                      Tập nước tối ưu  \
Thuật toán       Alpha-Beta Minimax                   Alpha-Beta   
Trạng thái                                                         
T1 — X sắp thắng          1       1                    (2, 6, 8)   
T2 — X cần hòa            0       0                         (7,)   
T3 — bất lợi             -1      -1           (1, 3, 4, 5, 6, 7)   
T4 — bàn rỗng             0       0  (0, 1, 2, 3, 4, 5, 6, 7, 8)   

                                               
Thuật toán                            Minimax  
Trạng thái                                     
T1 — X sắp thắng                    (2, 6, 8)  
T2 — X cần hòa                           (7,)  
T3 — bất lợi               (1, 3, 4, 5, 6, 7)  
T4 — bàn rỗng     (0, 1, 2, 3, 4, 5, 6, 7, 8)

## Alpha-Beta trước/sau move ordering

In [6]:
ordering_records = []
for name, board in STATES.items():
    for ordered in (False, True):
        found = exact_search(board, alpha_beta=True, move_ordering=ordered)
        ordering_records.append({"Trạng thái": name,
                                 "Move ordering": "centre→corners→edges" if ordered else "thứ tự chỉ số",
                                 "Nước đi": found["move"], "Giá trị": found["value"],
                                 "Số nút": found["nodes"], "Thời gian (ms)": round(found["elapsed_s"] * 1_000, 3)})
ordering = pd.DataFrame(ordering_records)
ordering

,Trạng thái,Move ordering,Nước đi,Giá trị,Số nút,Thời gian (ms)
0,T1 — X sắp thắng,thứ tự chỉ số,2,1,61,3.205
1,T1 — X sắp thắng,centre→corners→edges,2,1,44,2.280
2,T2 — X cần hòa,thứ tự chỉ số,7,0,101,5.995
3,T2 — X cần hòa,centre→corners→edges,7,0,149,7.576
4,T3 — bất lợi,thứ tự chỉ số,1,-1,238,12.048
5,T3 — bất lợi,centre→corners→edges,4,-1,51,2.394
6,T4 — bàn rỗng,thứ tự chỉ số,0,0,18297,875.534
7,T4 — bàn rỗng,centre→corners→edges,4,0,7275,363.398


### Nhận xét

Alpha-Beta giữ nguyên kết quả minimax vì chỉ cắt các nhánh không thể cải thiện giới hạn alpha/beta. Ordering ưu tiên ô giữa rồi góc thường giúp phát hiện giới hạn sớm hơn; mức cải thiện phụ thuộc trạng thái và không phải lúc nào thời gian đơn lẻ cũng giảm do nhiễu đo rất nhỏ.

In [7]:
summary = ordering.pivot(index="Trạng thái", columns="Move ordering", values="Số nút")
summary["Giảm node (%)"] = (1 - summary["centre→corners→edges"] / summary["thứ tự chỉ số"]) * 100
summary.round(1)

Move ordering,centre→corners→edges,thứ tự chỉ số,Giảm node (%)
Trạng thái,,,
T1 — X sắp thắng,44,61,27.9
T2 — X cần hòa,149,101,-47.5
T3 — bất lợi,51,238,78.6
T4 — bàn rỗng,7275,18297,60.2
